<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## ✅ M2 Lab 1 · 参考答案：guardrail + hook + memory

**一句话**：官方完整解答。代码与练习版逐行对应，这里只讲每一步「填的是什么、为什么这么填」，以及它和你的版本的差别。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ Solution</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M2_Lab_1_deep_research_p1_solution.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：C1M2_Lab_1_deep_research_p1.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📂 复用上一级的 config/ 与 utils.py</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：给 4-agent 研究 crew（Planner → Researcher → Fact Checker → Writer）加上可靠性：
最终报告的 **guardrail**、把报告存盘的 **after_kickoff hook**、以及 crew 级 **memory**。
完整概念讲解见练习 notebook；本 notebook 侧重「答案 + 对照」。

</div>

<img src="../images/lab1-agents-tasks-diagram.png" width="900">

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📌 和你的练习版对照

</div>

| 填空处 | 参考答案 | 你的答案 |
|---|---|---|
| guardrail 第三项检查 | 正则 `#+.*citations` 或 `#+.*references` 命中即可 | 相同 ✅（错误提示少写了 "(or References)"，不影响逻辑） |
| `write_final_report` | `guardrails=[write_report_guardrail]` | 相同 ✅ |
| Crew | `memory=True`, `after_kickoff_callbacks=[save_file_hook]` | 相同 ✅ |
| 运行 | `deep_research_crew.kickoff(...)` | `await ...kickoff_async(...)`（notebook 里必须这样，见下） |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**本 notebook 为能直接运行所做的改动**（Solution/ 是子文件夹，原代码按「放在 lab 根目录」写的）：
1. `sys.path.insert(0, "..")` → 才能 `from utils import ...`；
2. `config/...yaml` → `../config/...yaml`；
3. `kickoff()` → `await kickoff_async()`。
报告 `research_report.md` 和 memory 都会落在 **Solution/** 自己的位置，不会覆盖练习版的结果。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：导入 + 环境变量

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔑 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">utils.get_*_api_key()</code></div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">load_dotenv(find_dotenv())</code>：从当前目录往上找 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">.env</code>（找到仓库根的那份），再读环境变量。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🤖 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">os.environ[&quot;MODEL&quot;]</code></div><div style="font-size:0.92em; line-height:1.6;">没给 agent 传 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">llm=</code> 时，CrewAI 用 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">MODEL</code> 环境变量当默认模型（不设则用 crewai 内置默认，1.15.22 是 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">gpt-4.1-mini</code>）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧪 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">CREWAI_TESTING=true</code></div><div style="font-size:0.92em; line-height:1.6;">让 crewai 认为处于测试环境：<b>跳过首次运行时「是否开启 tracing」的交互确认</b>，notebook 不会卡住等输入。</div></div></div></div>

In [1]:
from crewai import Agent, Task, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
import sys
sys.path.insert(0, "..")   # 🆕 Solution/ 子目录：让 utils.py（在上一级）可被 import
os.environ["CREWAI_TESTING"] = "true"   # 跳过首次运行的 tracing 交互确认
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"   # 没指定 llm 的 agent 默认用它
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the EXA API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：工具 + 4 个 agent（与练习版相同）

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📄 配置文件：`config/agents.yaml` 与 `config/tasks.yaml`

</div>

agent 的 `role / goal / backstory` 和 task 的 `description / expected_output` 都写在 YAML 里，Python 只负责「装配」：

| 文件 | 顶层 key | 每个 key 下的字段 | Python 里怎么用 |
|---|---|---|---|
| `agents.yaml` | `research_planner` `internet_researcher` `fact_checker` `report_writer` | `role` `goal` `backstory` | `Agent(config=agent_config["research_planner"], ...)` |
| `tasks.yaml` | `create_research_plan` `gather_research_data` `verify_information_quality` `write_final_report` | `description` `expected_output` | `Task(config=task_config["..."], agent=...)` |

- `config=` 字典里的字段等价于直接写 `Agent(role=..., goal=...)`；工具、`verbose`、agent 绑定等 **运行时对象** YAML 表达不了，所以在 Python 里补。
- `{user_query}` 是占位符：`kickoff(inputs={"user_query": ...})` 时被替换进 **所有** agent / task 文本。
- YAML 语法：`'...'` 单引号字符串和 `>` 折叠块都可以跨行，换行会被折成空格。

两个课程视频里没讲、lab 里加上的参数（都是为了跑得快、别撞限流）：
- `max_rpm=150`：该 agent 每分钟最多 150 次 LLM 请求，超了自动等待；
- `max_iter=15`：思考↔调工具 最多 15 轮（默认 25），到上限必须给出「当前最好答案」。

In [2]:
# create the tool instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))    # deprecated 别名，仍可用（新名 ExaSearchTool）
scrape_website_tool = ScrapeWebsiteTool()

# load the configuration file for the agents  (🆕 ../ = lab 根目录)
with open('../config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)

# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],   # 只有 researcher / fact checker 能上网
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

/var/folders/xx/ypnl5f2n0y7b48w_pgxyhqt80000gn/T/ipykernel_35922/2063181936.py:2: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))    # deprecated 别名，仍可用（新名 ExaSearchTool）


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：guardrail —— 答案就是第三个 `re.search`

</div>

思路和前两项完全一样：Citations 和 References 都算「引用节」，所以用 `|` 两个都接受；`#+` 保证关键词出现在 Markdown 标题行里。
失败时返回的字符串会成为给 Report Writer 的修改意见，所以要写清楚「缺什么、应该长什么样」。

In [3]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw    # 运行时是 TaskOutput → 取 .raw；测试时直接传 str
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    ### START CODE HERE ###

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower):   # ← 答案：引用节两种叫法都接受
        return (False,
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )

    ### END CODE HERE ###
    return (True, output)

两份测试报告：第一份 → `(True, 原文)`；第二份 → `(False, '...Insights section...')`（第一个缺失项就返回）。

In [4]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [5]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：task —— 只给最后一个 task 挂 guardrail

</div>

为什么只挂在 `write_final_report`？检查的是「最终报告的结构」，中间 task（研究计划、原始资料）本来就不该有 Summary/Citations 这些节。

In [6]:
# load the configuration file for the tasks
with open('../config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

### START CODE HERE ###

# create the tasks using the configuration
create_research_plan = Task(
    config=task_config['create_research_plan'],
    agent=research_planner
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task(
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail],   # ← 答案：列表形式，可挂多个
)

### END CODE HERE ###

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 5 步：after_kickoff hook（给定代码）

</div>

`after_kickoff_callbacks` 里的函数在 **所有 task 结束后** 被调用，参数 `result` 是 `CrewOutput`：
`result.tasks_output` = 每个 task 的 `TaskOutput` 列表，`[-1].raw` 就是最终报告的 Markdown 原文。
（对应还有 `before_kickoff_callbacks`：kickoff 前调用，可以修改 `inputs`。）

In [7]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report.md"   # 相对 cwd → 存在 Solution/ 下
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

    return result   # 🆕 必须把 result 还回去：hook 的返回值会替换 kickoff 的返回值（原代码没有这行 → result 变成 None）

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 修正课程代码：hook 末尾加了 `return result`**
crewai 源码（1.15.22 `crew.py`）里是 `for cb in self.after_kickoff_callbacks: result = cb(result)` ——每个 after-kickoff 回调的 **返回值会替换** crew 的最终结果（字段说明："It may be used to adjust the output of the crew"）。
课程的 `save_file_hook` 只存文件、不返回 → `kickoff` 返回 `None`，后面再用 `result.raw` 就会 `AttributeError: 'NoneType' object has no attribute ...`（课程 notebook 恰好没再用 `result`，所以没暴露）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 6 步：Crew —— `memory=True` + `after_kickoff_callbacks`

</div>

`after_kickoff_callbacks` 必须是 **列表**（可以挂多个收尾函数，按顺序执行）；传函数本身，不要加括号调用。

In [8]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],

    ### START CODE HERE ###
    
    # add memory to the crew
    memory=True,   # ← 答案：启用统一 Memory
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]   # ← 答案：函数对象放进列表

    ### END CODE HERE ###
)

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026-10 版本差异：`memory=True` 背后是什么**（已对照 crewai 1.15.22 源码）
- 旧版课程时代：short-term / long-term / entity 三种记忆（ChromaDB + SQLite）。
- 现在：统一的 `Memory` 类（`crewai/memory/unified_memory.py`）——存储默认 **LanceDB**，embedding 默认 OpenAI `text-embedding-3-large`，存盘时还会用 agent 的 LLM 分析内容、打分类和重要度。
- 存在 `~/Library/Application Support/<notebook 所在文件夹名>/memory`（`crewai_core.paths.db_storage_path()` 用 `Path.cwd().name` 当 app 名；设 `CREWAI_STORAGE_DIR` 可改），**跨运行持久**：同一文件夹、同名 crew（默认 scope `/crew/crew`）再次运行会召回上次的记忆。
- 召回点：每个 task 开始前用 task 描述 `recall(limit=5)`，命中的记忆以 "Relevant memories:" 拼进 prompt。
- 还会给每个 agent **自动加两个工具** `Search memory` / `Save to memory`（`crewai/tools/memory_tools.py`），所以 verbose 日志里会看到 agent 自己调用 memory 工具（"Found memories" / "Saving N items to memory"）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 7 步：输入 + 运行

</div>

In [9]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
    "user_query": "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."   # ✏️ 改这里：研究问题（对应 YAML 里的 {user_query}）
}

### END CODE HERE ###

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异：`crew.kickoff()` → `await crew.kickoff_async()`**
原代码 `result = deep_research_crew.kickoff(inputs=inputs)` 在 notebook 里会报
`RuntimeError: Agent execution was invoked synchronously from within a running event loop`。
原因：Jupyter 内核本身一直跑着 asyncio 事件循环，crewai 1.15 的 agent 检测到它就拒绝同步执行。`kickoff_async` 在 1.15.22 里就是 `await asyncio.to_thread(self.kickoff, ...)`，把同步 kickoff 放到线程里跑，绕开事件循环。纯 `.py` 脚本里用 `kickoff()` 没问题。

</div>

In [10]:
# Execute the crew's tasks
result = await deep_research_crew.kickoff_async(inputs=inputs)   # 🆕 原为 deep_research_crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top five emerging AI tools for automating competitive        │
│  market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing   │
│  firm." into specific topics and key questions that need investigation. Create a focused research plan.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Research Plan for Evaluating Emerging AI Tools for Automating Competitive Market Analysis                  │
│                                                                                                                 │
│  #### 1. Main Research Topics to Investigate                                                                    │
│                                                                                                                 │
│  1. **Identification of Top Five Emerging AI Tools**                                                            │
│  2. **Features and Functionalities of Each Tool**                                                               │
│  3. **Limitations and Challenges**                                                                              │
│  4. **Cost Analysis**                                                                                           │
│  5. **Ideal Use Cases for Mid-sized Marketing Firms**                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### 2. Key Questions for Each Topic                                                                           │
│                                                                                                                 │
│  ##### Topic 1: Identification of Top Five Emerging AI Tools                                                    │
│  - What are the top five emerging AI tools currently available that automate competitive market analysis?       │
│  - What criteria were used to determine the "top" tools (e.g., market presence, user reviews, technological     │
│  advancements)?                                                                                                 │
│                                                                                                                 │
│  ##### Topic 2: Features and Functionalities of Each Tool                                                       │
│  - What are the key features of each of the identified AI tools?                                                │
│  - How do these tools integrate with existing marketing technologies and platforms?                             │
│  - What unique functionalities do each of these tools offer that differentiate them from competitors?           │
│                                                                                                                 │
│  ##### Topic 3: Limitations and Challenges                                                                      │
│  - What are the major limitations and challenges associated with each identified tool?                          │
│  - Are there specific industries or market scenarios where these tools may not be effective?                    │
│  - How do data privacy and ethical considerations affect the use of these tools?                                │
│                                                                                                                 │
│  ##### Topic 4: Cost Analysis                          

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool exa_search_tool executed with result: Title: The 10 best competitor analysis tools in 2026
URL: https://zapier.com/blog/competitor-analysis-tools/
ID: https://zapier.com/blog/competitor-analysis-tools/
Score: None
Published Date: 2026-09-...
Tool exa_search_tool executed with result: Title: Tools for AI Market Research: Turn Data Into Insights
URL: https://www.qualtrics.com/articles/strategy-research/ai-market-research/
ID: https://www.qualtrics.com/articles/strategy-research/ai-m...
Tool exa_search_tool executed with result: Title: The Complexity of Competitive Intelligence in the Age of data ambiguity and Artificial Intelligence
URL: https://doi.org/10.37380/jisib.v14.si1.2417
ID: https://exa.ai/library/publication/bln34...
Tool exa_search_tool executed with result: Title: Market Intelligence Platform Pricing: Full Cost Breakdown
URL: https://www.userintuition.ai/posts/market-intelligence-cost/
ID: https://www.userintuition.ai/posts/market-intelligence-cost/
Scor...
Tool exa_sea

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query "Evaluate the top five emerging AI tools   │
│  for automating competitive market analysis, including their features, limitations, costs, and ideal use cases  │
│  for a mid-sized marketing firm." using all verified research data. Structure it with clear sections, include   │
│  citations, and provide actionable insights.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.84) The research evaluated five emerging AI tools for automating competitive market analysis.
  categories: research, AI tools, market analysis
  entities: []
  dates: []
  ...
Tool search_memory executed with result: Found memories:
- (score=0.71) Outmano is an AI-powered competitor tracker that automates the research process by monitoring competitors' pricing, product launches, SEO changes, and reviews, starting ...
Tool search_memory executed with result: Found memories:
- (score=0.75) The research plan focuses on evaluating the top five emerging AI tools for automating competitive market analysis.
  categories: AI Tools, Market Analysis, Research Plan...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report: Evaluation of Top Five Emerging AI Tools for Automating Competitive Market Analysis    │
│  for Mid-Sized Marketing Firms                                                                                  │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│                                                                                                                 │
│  As the competitive landscape for businesses continues to evolve, mid-sized marketing firms increasingly seek   │
│  automated solutions to gather and analyze market intelligence. This report examines five emerging AI tools     │
│  designed to facilitate competitive market analysis. Each tool is evaluated based on its features,              │
│  limitations, costs, and ideal use cases, providing mid-sized firms with actionable insights into selecting     │
│  the right solution for their needs.                                                                            │
│                                                                                                                 │
│  The evaluated AI tools are:                                                                                    │
│  1. Outmano                                                                                                     │
│  2. Cohorva                                                                                                     │
│  3. MarketAgent                                                                                                 │
│  4. Agensio                                                                                                     │
│  5. Rivalize                                                                                                    │
│                                                                                                                 │
│  Overall, the adoption of these tools can enhance competitive intelligence, streamline processes, and allow     │
│  marketing firms to operate more efficiently.                                                                   │
│                                                                                                                 │
│  ## 2. Detailed Findings                                                                                        │
│                                                                                                                 │
│  ### 2.1. Outmano                                                                                               │
│                                                                                                                 │
│  - **Overview**: An AI-powered competitor tracker that automates the research process by monitoring             │
│  competitors' pricing, product launches, SEO changes, and reviews.                                              │
│  - **Features**:                                                                                                │
│    - Weekly briefings detailing competitor changes.                                                             │
│    - Multi-angle monitoring (pricing strategies, SEO tr

Report successfully saved to: research_report.md


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 看结果

</div>

In [11]:
from pathlib import Path
from IPython.display import display

for t in result.tasks_output:                       # 每个 task 的执行者与输出长度
    print(f"{t.agent:<22} {len(t.raw):>6} chars")
display(Markdown(Path("research_report.md").read_text()))   # hook 存下的报告（Solution/research_report.md）

Research Planner         2807 chars
Internet Researcher      5050 chars
Fact Checker             4007 chars
Report Writer            6568 chars


# Comprehensive Report: Evaluation of Top Five Emerging AI Tools for Automating Competitive Market Analysis for Mid-Sized Marketing Firms

## 1. Executive Summary

As the competitive landscape for businesses continues to evolve, mid-sized marketing firms increasingly seek automated solutions to gather and analyze market intelligence. This report examines five emerging AI tools designed to facilitate competitive market analysis. Each tool is evaluated based on its features, limitations, costs, and ideal use cases, providing mid-sized firms with actionable insights into selecting the right solution for their needs. 

The evaluated AI tools are:
1. Outmano
2. Cohorva
3. MarketAgent
4. Agensio
5. Rivalize

Overall, the adoption of these tools can enhance competitive intelligence, streamline processes, and allow marketing firms to operate more efficiently.

## 2. Detailed Findings

### 2.1. Outmano

- **Overview**: An AI-powered competitor tracker that automates the research process by monitoring competitors' pricing, product launches, SEO changes, and reviews.
- **Features**:
  - Weekly briefings detailing competitor changes.
  - Multi-angle monitoring (pricing strategies, SEO traffic, product releases).
  - Integration with AI agents like Claude and Codex for real-time insights.
- **Limitations**:
  - May require additional technical expertise for optimal integration.
  - Reliant on data quality for accuracy in insights.
- **Cost**: Starting at **$29.40/month**.
- **Ideal Use Cases**: Best for firms needing regular updates on competitive changes in fast-paced industries.
- **Website**: [Outmano](https://outmano.com/)

### 2.2. Cohorva

- **Overview**: An AI platform for competitive analysis that allows automated tracking of competitive gaps, pricing changes, ad monitoring, and digests.
- **Features**:
  - Custom AI agents tailored to specific competitors.
  - Automation of competitive research tasks.
  - Integration with external tools for enhanced data insights.
- **Limitations**:
  - Lack of clear pricing details may deter potential users; costs are flexible.
  - May not fully address niche market needs due to broad functionality.
- **Cost**: Not specified.
- **Ideal Use Cases**: Recommended for firms that require customization in monitoring specific competitive landscapes.
- **Website**: [Cohorva](https://cohorva.com/use-cases/competitor-analysis)

### 2.3. MarketAgent

- **Overview**: An AI market research platform that generates sourced reports on market trends and competitor activities.
- **Features**:
  - Automatic monitoring of competitors with weekly alerts.
  - Comprehensive competitive profiles and pricing comparisons.
- **Limitations**:
  - Can be more useful for larger datasets; might overwhelm smaller firms.
  - Alerts are only as valuable as the relevance of the chosen metrics.
- **Cost**: Starts from **$99/month** billed annually.
- **Ideal Use Cases**: Suitable for firms looking for detailed reports and in-depth analysis on multiple competitors.
- **Website**: [MarketAgent](https://marketagent.ai/)

### 2.4. Agensio

- **Overview**: A competitive intelligence agent for small and medium businesses providing comprehensive monitoring of competitors.
- **Features**:
  - Updates on competitor actions across 50+ sources.
  - Smart alerts for critical updates and market shifts.
  - Integrates with productivity tools for streamlined operations.
- **Limitations**:
  - Primarily targets the European market; functionality may vary by region.
  - Limited customization options compared to competitors.
- **Cost**: **€82.50/month** excluding tax.
- **Ideal Use Cases**: Best for firms focused on competitor monitoring across multiple channels and geographical locations.
- **Website**: [Agensio](https://agensio.ai/en/agent-veille/)

### 2.5. Rivalize

- **Overview**: Provides extensive insights into competitors with actionable recommendations, enabling firms to strategize effectively.
- **Features**:
  - Cited findings with actionable insights on competitor movements.
  - Comprehensive monitoring of various online competitor activities.
- **Limitations**:
  - Unclear pricing details may be a barrier; users may have to inquire directly.
  - Possibly less effective in very niche markets due to generalized analyses.
- **Cost**: Not specified.
- **Ideal Use Cases**: Well-suited for companies wanting to enhance their strategic responses to competitive actions.
- **Website**: [Rivalize](https://rivalize.ai/)

## 3. Supporting Evidence and Analysis

- **Cost Analysis**: All tools present varying pricing strategies, ranging from subscription models like Outmano and MarketAgent to custom pricing with tools like Cohorva and Rivalize, adding flexibility but potentially leading to unclear budgeting.
- **Integration Considerations**: Most tools can integrate with existing marketing frameworks, but firms need to evaluate compatibility with their current systems for maximum efficiency.
- **Limitations and Challenges**: Common challenges include data quality concerns, industry effectiveness variances, and privacy issues associated with automated data gathering.

## 4. Complete Source Citations

- Outmano. (n.d.). Retrieved from [Outmano](https://outmano.com/)
- Cohorva. (n.d.). Retrieved from [Cohorva](https://cohorva.com/use-cases/competitor-analysis)
- MarketAgent. (n.d.). Retrieved from [MarketAgent](https://marketagent.ai/)
- Agensio. (n.d.). Retrieved from [Agensio](https://agensio.ai/en/agent-veille/)
- Rivalize. (n.d.). Retrieved from [Rivalize](https://rivalize.ai/)

## 5. Key Insights and Recommendations

- Mid-sized marketing firms should prioritize tools that align closely with their operational needs and budget constraints. 
- It is recommended to take advantage of free trials (if available) for any of the tools prior to commitment.
- Continuous evaluation of data insights and feedback mechanisms will be crucial for adapting the chosen tools effectively to evolving market demands.
- Firms should consider potential hidden costs such as training and implementation to avoid budget overruns.

### Next Steps
1. Evaluate the tools based on specific business needs and target capabilities.
2. Consider pilot testing selected tools before full deployment.
3. Engage with vendors to clarify pricing and explore customization options.

In conclusion, the evaluated AI tools present promising solutions for automated competitive market analysis, enabling mid-sized marketing firms to enhance their analytical capabilities and stay competitive in their respective markets.

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| 在 Solution/ 里直接跑原代码 | `ModuleNotFoundError: utils` / `FileNotFoundError: config/agents.yaml` | 文件都在上一级：`sys.path.insert(0, "..")` + `../config/...` |
| `after_kickoff_callbacks=[save_file_hook()]` | 定义 crew 时就调用了一次，`result` 参数缺失报错 | 传函数对象，不要加括号 |
| notebook 里用 `kickoff()` | `RuntimeError: ... running event loop` | `await crew.kickoff_async(...)` |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. 为什么 guardrail 的正则要带 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">#+</code>？</b></summary>

要求关键词出现在 Markdown 标题行里（`## Summary`），而不是正文随便某处提到 summary。

</details>

<details><summary><b>2. 为什么 guardrail 只挂在 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">write_final_report</code> 上？</b></summary>

它检查的是最终报告的结构；中间 task 的输出（计划、原始资料、核查报告）不需要也不应该有这些节。

</details>

<details><summary><b>3. 参考答案和练习版代码有哪些实际差别？</b></summary>

逻辑完全一致；只是错误提示文案多了 "(or References)"、Crew 参数排版不同，以及原解答用同步 `kickoff()`（notebook 里会报错）。

</details>

<details><summary><b>4. Solution 与练习版的 memory 会互相影响吗？</b></summary>

不会。存储路径按 cwd 的文件夹名区分：练习版在 `.../05_improving-a-deep-research-crew/memory`，这里在 `.../Solution/memory`。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**三个填空 = 一行正则 + 一个 `guardrails=[...]` + 两个 Crew 参数**；可靠性来自「检查后打回重做」和「结束后自动收尾」，不需要改 agent 本身。

</div>